In [ ]:
import copy
import itertools
import os
import sys
sys.path.append("/workspace/mta_vision_transformers/")
from collections import OrderedDict
from typing import Any, Callable, Dict, List, Set, Tuple

import numpy as np
import einops
import torch
import torch.nn as nn
import torch.nn.functional as Fn
import torch.utils.data
from matplotlib import pyplot as plt
from tensordict import TensorDict
from torch.utils._pytree import tree_flatten

from core.monitor import Monitor
from infrastructure import utils
from infrastructure.settings import DEVICE, OUTPUT_DEVICE, DTYPE
from dataset.construct import ImageDataset
from dataset.library import DATASETS


dataset_name, n_classes = DATASETS["Common"][1]
OUTPUT_DIR = "experiments/adaptive"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
# Ocean: 901085904
# Rose: 100390212
torch.set_printoptions(linewidth=400, sci_mode=False)

In [ ]:
from dataset.evaluation import ImageTextDataset, run_retrieval_evaluation, print_retrieval_metrics, DEFAULT_DATASET
from modeling.image_features import ImageFeatures
from modeling.openclip_vit import OpenCLIPViT
from modeling.vit_adaptive import OpenCLIPAdaptiveViT


utils.reset_seed()
# Run evaluation
mode = "mask"
baseline_model = OpenCLIPViT().to(DEVICE)

# Evaluate base model
print("=" * 120)
print("Base model")
print("=" * 120)
baseline_fname = f"{OUTPUT_DIR}/metrics/baseline.pt"
if not os.path.exists(baseline_fname):
    baseline_metrics: TensorDict = run_retrieval_evaluation(baseline_model)
    torch.save(baseline_metrics, baseline_fname)
else:
    baseline_metrics: TensorDict = torch.load(baseline_fname, map_location=DEVICE)
print_retrieval_metrics(baseline_metrics)
print()

# Run adaptive model at least once to cache the masks
for extract_mode in ("MA", "AS"):
    mask_fname = f"experiments/saved_masks_COCO2017/{extract_mode}_mask.pt"
    if not os.path.exists(mask_fname):
        adaptive_model = OpenCLIPAdaptiveViT(mode, extract_mode, mask_layer=13, reset_layer=9, detection_layer=13)
        monitor = Monitor(adaptive_model, {"model.visual.transformer.return_mask": "mask"}, device=DEVICE)
        log = monitor.reset()
        run_retrieval_evaluation(adaptive_model)
        monitor.delete()
        torch.save(torch.cat(log["mask"], dim=0), mask_fname)
raise Exception()
# Evaluate adaptive model for each layer of removal
exclude_mask: torch.Tensor = torch.load(f"{OUTPUT_DIR}/AS_mask.pt", map_location=DEVICE)
mask_dict: Dict[str, torch.Tensor] = {
    k: torch.load(f"{OUTPUT_DIR}/{k}_mask.pt", map_location=DEVICE)
    for k in ("MA", "AS", "Artifact")
}

dataset: ImageTextDataset = copy.copy(DEFAULT_DATASET)
B: int = len(dataset)
for k, original_mask in mask_dict.items():
    print("=" * 120)
    print(f"{k} {mode} mode")
    print("=" * 120)
    
    indices = torch.argsort(exclude_mask[:, ImageFeatures.image_indices].to(torch.float) + torch.rand((B, ImageFeatures.N)), dim=1) + 1
    null_mask = torch.full_like(original_mask, False)
    null_mask[torch.arange(B)[:, None], indices] = (torch.arange(ImageFeatures.N) < torch.sum(original_mask, dim=1, keepdim=True))
    assert torch.all(torch.sum(null_mask, dim=1) == torch.sum(original_mask, dim=1)) and not torch.any(null_mask * exclude_mask)
    mask_list: List[torch.Tensor] = [original_mask, null_mask]

    save_fname = f"{OUTPUT_DIR}/metrics/{k}_{mode}.pt"
    if os.path.exists(save_fname):
        _: Tuple[torch.Tensor, TensorDict] = torch.load(save_fname, map_location=DEVICE)
        done, result = _
    else:
        done: torch.Tensor = torch.full((len(mask_list), ImageFeatures.NUM_LAYERS), False)
        result: TensorDict = baseline_metrics.expand((len(mask_list), ImageFeatures.NUM_LAYERS))
        
    for (mask_idx, mask), layer_idx in itertools.product(enumerate(mask_list), range(ImageFeatures.NUM_LAYERS)):
        if done[mask_idx, layer_idx]:
            continue
        
        print(f"Mask index: {mask_idx}, layer index: {layer_idx}")
        model = OpenCLIPAdaptiveViT(mode, extract_mode, mask_layer=layer_idx, reset_layer=9, detection_layer=13)
        dataset.load_cache({"mask": mask})
        
        done[mask_idx, layer_idx] = True
        result[mask_idx, layer_idx] = run_retrieval_evaluation(model, dataset=dataset, **evaluation_kwargs)
        
        torch.save((done, result), save_fname)
        print(f"\t{os.path.getsize(save_fname)} bytes written to {save_fname}")
        utils.empty_cache()